In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [2]:
import torch
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from models.regression.tcn import TemporalConvolutionalNetwork as TCN
from models.regression.lstm import LongShortTermMemory as LSTM
from models.regression.mlp import MultiLayerPerceptron as MLP

from loaders._load_vn30_reg import preprocess as preprocess_v1
from loaders._load_vn30_reg_deep import preprocess as preprocess_v2
from loaders._load_vn30_meta import VN30

from sklearn.metrics import r2_score, mean_absolute_percentage_error
from sklearn.linear_model import MultiTaskLassoCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

In [4]:
def create_meta_features(model, name: str, symbol: str):
    if name in ['tcn', 'lstm']:
        model.load_state_dict(torch.load(f'checkpoints_{name}/{name}_{symbol}.pth', map_location=torch.device('cpu')))
    model.eval()

    _, valid_loader, test_loader, scaler = preprocess_v2(symbol, mode=name, val=0.2)

    valid_features = []
    test_features = []
    with torch.no_grad():
        for X_batch, _ in valid_loader:
            X_batch = X_batch
            preds = model(X_batch).cpu().numpy()
            valid_features.append(preds)
        for X_batch, _ in test_loader:
            X_batch = X_batch
            labels = model(X_batch).cpu().numpy()
            test_features.append(labels)
    valid_features = np.vstack(valid_features)   # (n_samples, 4)
    test_features = np.vstack(test_features) # (n_samples, 4)
    valid_features = scaler.inverse_transform(valid_features)
    test_features = scaler.inverse_transform(test_features)

    return valid_features, test_features

In [23]:
tracks = {"r2": [], "mape": []}
for symbol in VN30:
    pack = preprocess_v1(symbol, val=0.2)
    X_val, Y_val = pack['val']
    X_test, Y_test = pack['test']
    target_scaler = pack['scaler']['target']
    Y_val = target_scaler.inverse_transform(Y_val)
    Y_test = target_scaler.inverse_transform(Y_test)

    valid_feat_tcn, test_feat_tcn = create_meta_features(model=TCN(), name='tcn', symbol=symbol)
    valid_feat_lstm, test_feat_lstm = create_meta_features(model=LSTM(), name='lstm', symbol=symbol)

    all_outputs = np.concatenate([valid_feat_tcn, valid_feat_lstm, X_val], axis=1)
    all_labels = np.concatenate([test_feat_tcn, test_feat_lstm, X_test], axis=1)

    meta_feature_scaler = StandardScaler()
    meta_feature = meta_feature_scaler.fit_transform(all_outputs)
    meta_feature_test = meta_feature_scaler.transform(all_labels)

    meta_target_scaler = StandardScaler()
    meta_target = meta_target_scaler.fit_transform(Y_val)
    meta_target_test = meta_target_scaler.transform(Y_test)

    tscv = TimeSeriesSplit(n_splits=3)
    model = MultiTaskLassoCV(cv=tscv, n_alphas=200)
    model.fit(meta_feature, meta_target)

    meta_preds_test = model.predict(meta_feature_test)
    meta_preds_test = meta_target_scaler.inverse_transform(meta_preds_test)

    r2 = r2_score(Y_test, meta_preds_test)
    mape = mean_absolute_percentage_error(Y_test, meta_preds_test) * 100

    print(f"Symbol: {symbol}, Test R²: {r2:.4f}, Test MAPE: {mape:.4f}%")
    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

print(f"Mean R²: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}%")
print(f"Std R²: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}%")

Symbol: ACB, Test R²: 0.9457, Test MAPE: 0.8455%
Symbol: BCM, Test R²: 0.9672, Test MAPE: 1.0661%
Symbol: BID, Test R²: 0.8875, Test MAPE: 1.2592%
Symbol: BVH, Test R²: 0.9818, Test MAPE: 0.9180%
Symbol: CTG, Test R²: 0.9427, Test MAPE: 1.5898%
Symbol: FPT, Test R²: 0.9919, Test MAPE: 1.0458%
Symbol: GAS, Test R²: 0.9578, Test MAPE: 0.7544%
Symbol: GVR, Test R²: 0.9759, Test MAPE: 1.4061%
Symbol: HDB, Test R²: 0.9766, Test MAPE: 1.0570%
Symbol: HPG, Test R²: 0.9272, Test MAPE: 0.8675%
Symbol: LPB, Test R²: 0.9969, Test MAPE: 0.9866%
Symbol: MBB, Test R²: 0.9560, Test MAPE: 1.1510%
Symbol: MSN, Test R²: 0.9675, Test MAPE: 0.9412%
Symbol: MWG, Test R²: 0.9827, Test MAPE: 1.2368%
Symbol: PLX, Test R²: 0.9833, Test MAPE: 0.9698%
Symbol: SAB, Test R²: 0.9433, Test MAPE: 0.8824%
Symbol: SHB, Test R²: 0.9723, Test MAPE: 0.7993%
Symbol: SSB, Test R²: 0.9549, Test MAPE: 0.9300%
Symbol: SSI, Test R²: 0.9475, Test MAPE: 1.0452%
Symbol: STB, Test R²: 0.9840, Test MAPE: 0.9457%
Symbol: TCB, Test R²

In [6]:
tracks = {"r2": [], "mape": []}
for symbol in VN30:
    pack = preprocess_v1(symbol, val=0.2)
    X_val, Y_val = pack['val']
    X_test, Y_test = pack['test']
    target_scaler = pack['scaler']['target']
    Y_val = target_scaler.inverse_transform(Y_val)
    Y_test = target_scaler.inverse_transform(Y_test)

    valid_feat_tcn, test_feat_tcn = create_meta_features(model=TCN(), name='tcn', symbol=symbol)
    valid_feat_lstm, test_feat_lstm = create_meta_features(model=LSTM(), name='lstm', symbol=symbol)
    
    # For MLP
    ckpt = torch.load(f"checkpoints_mlp/mlp_{symbol}.pth", map_location="cpu")
    cfg = ckpt["config"]

    model = MLP(
        input_dim=cfg["input_dim"],
        output_dim=cfg["output_dim"],
        hidden_dim=cfg["hidden_dim"],
        num_layers=cfg["num_layers"],
        skip_bits=cfg["skip_bits"],
        dropout=cfg["dropout"],
        activation=cfg["activation"],
    )
    model.load_state_dict(ckpt["model_state_dict"])
    valid_feat_mlp, test_feat_mlp = create_meta_features(model=model, name='mlp', symbol=symbol)

    all_outputs = np.concatenate([valid_feat_tcn, valid_feat_mlp, X_val], axis=1)
    all_labels = np.concatenate([test_feat_tcn, test_feat_mlp, X_test], axis=1)

    meta_feature_scaler = StandardScaler()
    meta_feature = meta_feature_scaler.fit_transform(all_outputs)
    meta_feature_test = meta_feature_scaler.transform(all_labels)

    meta_target_scaler = StandardScaler()
    meta_target = meta_target_scaler.fit_transform(Y_val)
    meta_target_test = meta_target_scaler.transform(Y_test)

    tscv = TimeSeriesSplit(n_splits=3)
    model = MultiTaskLassoCV(cv=tscv, n_alphas=200)
    model.fit(meta_feature, meta_target)

    meta_preds_test = model.predict(meta_feature_test)
    meta_preds_test = meta_target_scaler.inverse_transform(meta_preds_test)

    r2 = r2_score(Y_test, meta_preds_test)
    mape = mean_absolute_percentage_error(Y_test, meta_preds_test) * 100

    print(f"Symbol: {symbol}, Test R²: {r2:.4f}, Test MAPE: {mape:.4f}%")
    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

print(f"Mean R²: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}%")
print(f"Std R²: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}%")

Symbol: ACB, Test R²: 0.9457, Test MAPE: 0.8408%
Symbol: BCM, Test R²: 0.9674, Test MAPE: 1.0651%
Symbol: BID, Test R²: 0.8876, Test MAPE: 1.2585%
Symbol: BVH, Test R²: 0.9817, Test MAPE: 0.9180%
Symbol: CTG, Test R²: 0.9418, Test MAPE: 1.6044%
Symbol: FPT, Test R²: 0.9919, Test MAPE: 1.0449%
Symbol: GAS, Test R²: 0.9574, Test MAPE: 0.7599%
Symbol: GVR, Test R²: 0.9759, Test MAPE: 1.4061%
Symbol: HDB, Test R²: 0.9803, Test MAPE: 0.9251%
Symbol: HPG, Test R²: 0.9279, Test MAPE: 0.8618%
Symbol: LPB, Test R²: 0.9969, Test MAPE: 0.9869%
Symbol: MBB, Test R²: 0.9551, Test MAPE: 1.1698%
Symbol: MSN, Test R²: 0.9675, Test MAPE: 0.9412%
Symbol: MWG, Test R²: 0.9826, Test MAPE: 1.2371%
Symbol: PLX, Test R²: 0.9833, Test MAPE: 0.9700%
Symbol: SAB, Test R²: 0.9391, Test MAPE: 0.9437%
Symbol: SHB, Test R²: 0.9724, Test MAPE: 0.7997%
Symbol: SSB, Test R²: 0.9587, Test MAPE: 0.8716%
Symbol: SSI, Test R²: 0.9471, Test MAPE: 1.0507%
Symbol: STB, Test R²: 0.9840, Test MAPE: 0.9466%
Symbol: TCB, Test R²